In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# output_file = "llama3.1_result.jsonl"
output_file = "..results/llama8b_r1_result.jsonl"

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

def load_model(model_name, device="auto"):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = LLM(
        model=model_name,
        tensor_parallel_size=8,  # set >1 if using multiple GPUs
        dtype="bfloat16",      
        gpu_memory_utilization=1.0
)
    return model, tokenizer

model, tokenizer = load_model(model_name)

INFO 08-24 02:49:18 [__init__.py:239] Automatically detected platform cuda.
INFO 08-24 02:49:27 [config.py:600] This model supports multiple tasks: {'score', 'reward', 'embed', 'generate', 'classify'}. Defaulting to 'generate'.
INFO 08-24 02:49:27 [config.py:1600] Defaulting to use mp for distributed inference
INFO 08-24 02:49:27 [config.py:1780] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-24 02:49:29 [core.py:61] Initializing a V1 LLM engine (v0.8.3) with config: model='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(VllmWorker rank=1 pid=1979639) INFO 08-24 02:49:37 [loader.py:447] Loading weights took 1.14 seconds
(VllmWorker rank=5 pid=1979732) INFO 08-24 02:49:37 [loader.py:447] Loading weights took 1.07 seconds
(VllmWorker rank=7 pid=1979785) INFO 08-24 02:49:37 [loader.py:447] Loading weights took 1.18 seconds
(VllmWorker rank=3 pid=1979688) INFO 08-24 02:49:37 [loader.py:447] Loading weights took 1.18 seconds
(VllmWorker rank=4 pid=1979712) INFO 08-24 02:49:37 [loader.py:447] Loading weights took 1.08 seconds
(VllmWorker rank=1 pid=1979639) INFO 08-24 02:49:37 [gpu_model_runner.py:1273] Model loading took 1.9029 GiB and 1.570516 seconds
(VllmWorker rank=5 pid=1979732) INFO 08-24 02:49:37 [gpu_model_runner.py:1273] Model loading took 1.9029 GiB and 1.596886 seconds
(VllmWorker rank=2 pid=1979662) INFO 08-24 02:49:37 [loader.py:447] Loading weights took 1.17 seconds
(VllmWorker rank=7 pid=1979785) INFO 08-24 02:49:37 [gpu_model_runner.py:1273] Model loading took 1.9029 GiB and 1.822957 second

In [3]:
# Text Generation
if "Llama" in model_name:
    BOS = 128000
    USER = 128011
    ASSISTANT = 128012
    NEWLINE = 198
    THINK_START = 128013
    THINK_END = 128014
    EOS = 128001
elif "Qwen" in model_name:
    BOS = 151646
    USER = 151644
    ASSISTANT = 151645
    NEWLINE = 198
    THINK_START = 151648
    THINK_END = 151649
    EOS = 151643
else:
    raise ValueError(f"Unknown tokens for model {model_name}")

In [4]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

def pprint(text):
    """Pretty print the model's generated text using rich."""
    console = Console(width=100)
    
    
    # Create markdown and display in a panel
    #md = Markdown(text.strip())
    console.print(Panel(text, border_style="blue"))


In [5]:
# load math 500 dataset: HuggingFaceH4/MATH-500
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/MATH-500")

if "R1" in model_name:
    def prompt_from_example(example, tokenizer):
        user_message = example['problem']
        math_suffix = " Please reason step by step, and put your final answer within \\boxed{}."
        toks = [BOS] + [USER] + tokenizer.encode(user_message+math_suffix, add_special_tokens=False) + [ASSISTANT] + [THINK_START] + [NEWLINE]
        #toks = [BOS] + tokenizer.encode(user_message+math_suffix, add_special_tokens=False) + [THINK_START] + [NEWLINE]
        return toks, tokenizer.decode(toks, skip_special_tokens=False)
else:
    def prompt_from_example(example, tokenizer):
        user_message = example['problem']
        math_suffix = " Please reason step by step, and put your final answer within \\boxed{}."
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message + math_suffix},
        ]
        toks = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        return toks, tokenizer.decode(toks, skip_special_tokens=False)

In [6]:
toks, prompt = prompt_from_example(dataset['test'][0], tokenizer)
answer = dataset['test'][0]['answer']

In [7]:
tokenizer

LlamaTokenizerFast(name_or_path='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', vocab_size=128000, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜end▁of▁sentence｜>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	128000: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=T

In [8]:
# generate a continuation of prompt using vllm

if "R1" in model_name:
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.95,
        max_tokens=32768
    )
else:
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.95,
        max_tokens=15000
    )

In [9]:
out = model.generate([prompt], sampling_params)
generated_text = out[0].outputs[0].text
print(generated_text)

Processed prompts: 100%|██████████| 1/1 [00:19<00:00, 19.06s/it, est. speed input: 3.57 toks/s, output: 77.27 toks/s]

Okay, so I need to convert the rectangular coordinate (0, 3) to polar coordinates. Hmm, I remember that polar coordinates are represented as (r, θ), where r is the distance from the origin and θ is the angle measured from the positive x-axis. Right? Let me try to recall the formulas to convert between rectangular and polar coordinates.

I think the formulas are:
- r = √(x² + y²)
- θ = arctan(y/x)

But wait, I should make sure I have the right formulas. Let me think. So, in rectangular coordinates, x is the horizontal distance from the origin, and y is the vertical distance. In polar coordinates, r is the radius, or the straight-line distance from the origin to the point, and θ is the angle you have to rotate from the positive x-axis to reach that point.

So, for the given point (0, 3), let me plug the values into the formulas. The x-coordinate is 0 and the y-coordinate is 3. 

First, calculating r. That's straightforward, right? r = √(0² + 3²) = √(0 + 9) = √9 = 3. Okay, so r is 3. That

In [10]:
import re

import sys
sys.path.append("/disk/u/troitskiid/projects/r1helpers")
from src.r1helpers.math500.grader import grade_answer

if "R1" in model_name:
    def parse_answer(generated_text):
        matches = re.search("</think>", generated_text)
        if matches is None:
            return ""
        generated_answer = generated_text[matches.end():]
        # in generated answer select the content of \boxed{}
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer
else:
    def parse_answer(generated_text):
        end_idx = generated_text.find("<|start_header_id|>assistant<|end_header_id|>")
        generated_answer = generated_text[end_idx:]
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer

parsed_answer = parse_answer(generated_text)
print(parsed_answer)
print(parsed_answer, answer)
grade_answer(parsed_answer, answer)

(3, \frac{\pi}{2})}
(3, \frac{\pi}{2})} \left( 3, \frac{\pi}{2} \right)


True

In [11]:
# evaluate in batches and store results in both dataframe and jsonl file
import pandas as pd
import json
from tqdm import tqdm

df = pd.DataFrame(columns=['problem', 'answer', 'generated_answer', 'correct'])
n_correct = 0

batch_size = 8
examples = list(dataset['test'])

for batch_start in tqdm(range(0, len(examples), batch_size)):
    batch_examples = examples[batch_start:batch_start+batch_size]
    prompts = []
    for example in batch_examples:
        toks, prompt = prompt_from_example(example, tokenizer)
        prompts.append(prompt)
    outs = model.generate(prompts, sampling_params)
    for i, (example, out) in enumerate(zip(batch_examples, outs)):
        generated_text = out.outputs[0].text
        generated_answer = parse_answer(generated_text)
        example['parsed_answer'] = generated_answer
        example['correct'] = grade_answer(generated_answer, example['answer'])
        example['generated_text'] = generated_text
        n_correct += int(example['correct'])
        idx = batch_start + i
        print(n_correct / (idx + 1))

        # add to both dataframe and jsonl file
        df = pd.concat([df, pd.DataFrame([example])], ignore_index=True)

        # Write to jsonl file immediately after each example
        with open(output_file, 'a') as f:
            f.write(json.dumps(example) + '\n')
            f.flush() # Ensure it's written to disk


  0%|          | 0/63 [00:00<?, ?it/s]

  2%|▏         | 1/63 [01:10<1:12:28, 70.13s/it]

1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.875


  3%|▎         | 2/63 [05:42<3:12:13, 189.07s/it]

0.8888888888888888
0.8
0.8181818181818182
0.75
0.7692307692307693
0.7857142857142857
0.8
0.8125


...

0.8814968814968815
0.8817427385892116
0.8819875776397516
0.8822314049586777
0.8824742268041237
0.8827160493827161
0.8829568788501027
0.8831967213114754


 98%|█████████▊| 62/63 [3:46:46<03:48, 228.57s/it]

0.8834355828220859
0.8836734693877552
0.8839103869653768
0.8841463414634146
0.8843813387423936
0.8846153846153846
0.8848484848484849
0.8850806451612904


100%|██████████| 63/63 [3:50:47<00:00, 219.80s/it]

0.8853118712273642
0.8855421686746988
0.8857715430861723
0.886
